# 02 - Figures

Draws every figure of the report from `output/` and the tables of notebook 01 into `figures/` (run `01_summary_tables.ipynb` first).
Needs numpy, pandas and matplotlib. A figure whose inputs are missing is skipped with a message.

In [ ]:
import os, sys
from pathlib import Path
from types import SimpleNamespace

# Work from the repository root (the directory that holds the Makefile), wherever the notebook is opened from.
REPO = Path.cwd().resolve()
while not (REPO / "Makefile").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "Makefile").exists(), "open this notebook from inside the repository (the directory with the Makefile)"
os.chdir(REPO)

ROOT      = "output"                 # raw experiment results (the results_* directories of implementation/scripts/submit_all.sh)
DATA_REAL = "output/data_real"       # cell-type labels of the real data set (Tabula Sapiens)
TABLES    = "work/csv"             # summary tables written by summarize.py
SRC       = "work/source"          # source tables of the figures and of the LaTeX tables
TEX       = "work/tex"             # the LaTeX tables of the report
FIGS      = "figures"                # the figures of the report
WORK      = "work/results"           # raw files arranged by report section (intermediate, regenerated by notebook 01)
for d in (TABLES, SRC, TEX, FIGS, WORK):
    os.makedirs(d, exist_ok=True)

## 1. Figures R1-R19, fig5, fig11

### Figures R1-R19, fig5, fig11

All figures of the report (R1-R19, plus the real-data scaling and cost-model figures), one message per figure. Inputs: the result directories in `output/` (CPU and GPU runs), the tables of notebook 01 and the real-data label files `output/data_real/`. A figure whose inputs are missing is skipped with a message. Requires numpy, pandas, matplotlib.

In [ ]:
import argparse, glob, json, os, re, statistics as st
A=SimpleNamespace(cpu=ROOT,gpu=ROOT,data=DATA_REAL,tables=TABLES,out=FIGS)   # CPU and GPU result directories are both under output/
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
plt.rcParams.update({"axes.axisbelow":True,"figure.dpi":130,"savefig.dpi":170,"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.color":"#e4e3df","grid.linewidth":0.7,
  "font.size":11,"axes.titlesize":12,"axes.titleweight":"bold","legend.fontsize":10.5,"legend.frameon":False,"axes.edgecolor":"#52514e","axes.unicode_minus":False})
# Stage palette (one fixed colour per iteration stage, used in every stage figure) and neutral colours for series that are not stages
# (problem sizes, variants, methods), so that no colour means two different things in the report.
STG=dict(fft="#2a78d6",kernel="#6250d6",spread="#1baf7a",attr="#eb6834",update="#c9c7c1",gsum="#e34948",trans="#e87ba4",allg="#eda100",exch="#8a9a1f")
NEU=dict(k="#222222",g="#8a8984",lg="#c9c7c1",br="#b5651d")
C=dict(blue="#2a78d6",orange="#eb6834",aqua="#1baf7a",yellow="#eda100",magenta="#e87ba4",violet="#6250d6",red="#e34948",gray="#8a8984",dark="#222222")
OUT=A.out; os.makedirs(OUT,exist_ok=True); CPU=A.cpu; GPU=A.gpu; TAB=A.tables
# Width of each figure in the report as a fraction of the text width; the canvas is shrunk to this width (the text is 6.3 in wide, so a figure of 8 in is shown at 0.8 of its size)
# so that the fonts keep their size in the PDF instead of being scaled down with the figure.
INC=dict(R1a_serial_anatomy=1.0,R1b_serial_N_scaling=0.8,R2_strong_scaling=1.0,R3_cpu_stage_bars=1.0,R4_placement_bars=1.0,R5_exchange_variants=1.0,R6_weak_scaling=1.0,R19_weak_stages=0.9,R7_gpu_stage_bars=0.85,R8_gpu_transport=0.8,R9_official_compare=1.0,R10_gpu_vs_cpu=0.9,R11_crossover=1.0,R12_kl_gap=1.0,R13_ablation_order_gridcomm=1.0,R14_interp_order=1.0,R15_real_embedding=0.85,R16_type_purity=0.85,R17_real_cpu_scaling=1.0,R18_cost_model=1.0,fig5_exact_validation=1.0,fig11_kscan=1.0)
ASPECT=dict()
SIZE=dict(R1a_serial_anatomy=(8,3.4),R1b_serial_N_scaling=(6.5,4.2),R2_strong_scaling=(10,5.0),R3_cpu_stage_bars=(8,4.6),R4_placement_bars=(10,4.6),R5_exchange_variants=(10,4.0),
 R6_weak_scaling=(9,4.0),R19_weak_stages=(7.6,4.4),R7_gpu_stage_bars=(7.5,4.4),R8_gpu_transport=(6.6,4.2),R9_official_compare=(9.5,4.3),R10_gpu_vs_cpu=(7.5,4.2),R11_crossover=(9.5,4.1),R12_kl_gap=(10,4.3),
 R13_ablation_order_gridcomm=(10,4.1),R14_interp_order=(9.5,4.0),R18_cost_model=(10,4.0),R17_real_cpu_scaling=(9,3.9),fig5_exact_validation=(9.5,4.1),fig11_kscan=(9.5,3.9))
PANEL=dict(R2_strong_scaling=["(a) Time per iteration","(b) Speed-up","(c) Efficiency"],R5_exchange_variants=["P=16","P=64","P=128 (5 repeats)"],R6_weak_scaling=["(a) Time per iteration","(b) Efficiency"],
 R11_crossover=["(a) Whole runs","(b) Time per iteration"],R13_ablation_order_gridcomm=["(a) Random blocks","(b) Non-blocking grid sum"],R14_interp_order=["(a) Gradient error","(b) Time per iteration"],
 fig5_exact_validation=["(a) Gradient error","(b) Parallel vs. serial"])
NOTITLE={"R1a_serial_anatomy","R1b_serial_N_scaling","R3_cpu_stage_bars","R4_placement_bars","R7_gpu_stage_bars","R8_gpu_transport","R10_gpu_vs_cpu"}
SMALLX={"R4_placement_bars":9,"R12_kl_gap":9.5,"fig5_exact_validation":9.5}
def shrink(fig,name):
    # The report shows a figure of 6.3 in text width; the canvas is made about as wide as the figure is printed (plus 1/3), so that the
    # fonts are not scaled down. Titles that only repeat the caption are removed and panel titles are kept short.
    if getattr(fig,"_suptitle",None) is not None: fig._suptitle.set_text("")
    axs=[a for a in fig.axes if a.get_title() or a.has_data()]
    if name in NOTITLE: axs[0].set_title("")
    if name in PANEL:
        for a,t in zip([a for a in axs if a.get_title()],PANEL[name]): a.set_title(t)
    if name in SMALLX:
        for a in axs:
            for t in a.get_xticklabels(): t.set_fontsize(SMALLX[name]); t.set_rotation(25 if name!="R4_placement_bars" else 0); t.set_ha("right" if name!="R4_placement_bars" else "center")
            for t in a.texts: t.set_fontsize(8.5)
    if name in SIZE: fig.set_size_inches(*SIZE[name])
    else:
        w,h=fig.get_size_inches(); W=8.0*INC.get(name,1.0)
        if w>W: fig.set_size_inches(W,h*W/w*1.15)
def save(fig,name): shrink(fig,name); fig.tight_layout(); fig.savefig(f"{OUT}/{name}.png"); plt.close(fig); print("wrote",name)
def num(s): return float(str(s).split()[0])
def pticks(ax,P): ax.set_xscale("log",base=2); ax.set_xticks(P); ax.set_xticklabels([str(p) for p in P]); ax.minorticks_off()
plain=FuncFormatter(lambda v,_:f"{v:g}")

In [ ]:
# ---- R1a serial anatomy -------------------------------------------------------------------------------------------
b=pd.read_csv(f"{CPU}/results_vanda_cpu_v2/serial/base.csv"); g=b.groupby("boxes").median(numeric_only=True).loc[[200,400,512]]
parts=[("med_fft","FFT convolution",STG["fft"]),("med_kernel_spec","FFT of the kernel",STG["kernel"]),("sg","spread + gather",STG["spread"]),("med_attraction","sparse attraction",STG["attr"]),("med_update","update",STG["update"])]
g["sg"]=g.med_spread+g.med_gather
fig,ax=plt.subplots(figsize=(9,3.4)); left=np.zeros(3)
for k,lab,col in parts:
    v=(g[k]/g.med_total*100).values; ax.barh(range(3),v,left=left,color=col,label=lab,height=0.6)
    for i,(x,w) in enumerate(zip(left,v)):
        if w>6: ax.text(x+w/2,i,f"{w:.0f}%",ha="center",va="center",color="white",fontsize=9,fontweight="bold")
    left+=v
ax.set_yticks(range(3)); ax.set_yticklabels([f"{int(n)} boxes\n({t:.2f} s / iteration)" for n,t in zip(g.index,g.med_total)]); ax.set_xlim(0,100); ax.set_xlabel("share of one serial iteration (%)"); ax.invert_yaxis()
ax.set_title("Serial iteration (250k points, 1 core): FFT work is 94-99% of the time"); ax.legend(ncol=3,fontsize=10,loc="upper center",bbox_to_anchor=(0.5,-0.2)); ax.grid(axis="y",visible=False)
save(fig,"R1a_serial_anatomy")

In [ ]:
# ---- R1b N scaling --------------------------------------------------------------------------------------------------
n=pd.read_csv(f"{CPU}/results_vanda_cpu_v2/serial/nscale.csv").groupby("N").median(numeric_only=True).reset_index()
sl=np.polyfit(np.log(n.N),np.log(n.med_total),1)[0]
fig,ax=plt.subplots(figsize=(7.2,4.4))
ax.plot(n.N,n.med_total,"o-",color=NEU["k"],label=f"whole iteration (slope {sl:.2f})"); ax.plot(n.N,n.med_fft,"s--",color=STG["fft"],label="FFT convolution"); ax.plot(n.N,n.med_attraction,"^-",color=STG["attr"],label="sparse attraction")
ref=n.med_total.values[0]*(n.N.values/n.N.values[0])*(np.log(n.N.values)/np.log(n.N.values[0])); ax.plot(n.N,ref,":",color=C["gray"],label=r"$N\log N$ reference")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("number of points N (grid area grows with N)"); ax.set_ylabel("time per iteration (s)"); ax.legend(fontsize=10)
ax.set_title("Serial cost grows as N log N when the grid grows with N"); save(fig,"R1b_serial_N_scaling")

In [ ]:
# ---- R2 strong scaling ------------------------------------------------------------------------------------------------
def Esum(d,iters):
    df=pd.read_csv(f"{CPU}/{d}/E_summary.csv"); df["t"]=df.loop_s_median/iters; df["s"]=df.speedup_vs_rep_P1.map(num); return df
S400=Esum("results_vcpu_E400_big",100); S512=Esum("results_vcpu_E512_big",100); S768=Esum("results_vcpu_E768_1m",60); R400=Esum("results_vcpu_E400",100); R512=Esum("results_vcpu_E512",100)
series=[("250k points, 400 boxes",S400,NEU["k"],R400),("250k points, 512 boxes",S512,NEU["g"],R512),("1M points, 768 boxes",S768,NEU["br"],None)]
plt.rcParams.update({"font.size":12.5,"axes.titlesize":12.5,"axes.labelsize":12.5,"xtick.labelsize":11.5,"ytick.labelsize":11.5,"legend.fontsize":11.5})
fig,ax=plt.subplots(1,3,figsize=(10.4,5.5))
for lab,df,col,rep in series:
    d=df[df.variant=="fftdist"]; p1=df[(df.variant=="fftrep")&(df.P==1)].iloc[0]
    P=[1]+list(d.P); T=[p1.t]+list(d.t); Sp=[1.0]+list(d.s); Ef=[1.0]+list(d.s/d.P)
    pk=d.loc[d.s.idxmax()]; ax[0].plot(P,T,"o-",color=col,ms=4); ax[1].plot(P,Sp,"o-",color=col,ms=4); ax[2].plot(P,Ef,"o-",color=col,ms=4)
    if rep is not None:
        r=rep[rep.variant=="fftrep"]; ax[0].plot(r.P,r.t,"s--",color=col,alpha=.8,ms=3.5,lw=1.2); ax[1].plot(r.P,r.s,"s--",color=col,alpha=.8,ms=3.5,lw=1.2); ax[2].plot(r.P,r.s/r.P,"s--",color=col,alpha=.8,ms=3.5,lw=1.2)
    e=d[d.P==512]; ax[2].annotate(f"{(e.s/e.P).iloc[0]:.2f}",(512,(e.s/e.P).iloc[0]),xytext=(5,-3),textcoords="offset points",fontsize=10.5,color=col,ha="left")
# evenly filled nodes (placement jobs, 250k points, 400 boxes); T(1) from the paired job of the strong scaling
pp=pd.concat([pd.read_csv(f"{CPU}/results_place256/place.csv"),pd.read_csv(f"{CPU}/results_place8/place.csv")]); pm=pp.groupby(["P","K"]).sec_per_iter_steady.median()
ev=[(144,48),(144,72),(216,54),(216,72),(256,64)]; Pe=np.array([a for a,b in ev]); Te=np.array([pm[k] for k in ev]); t1=S400[(S400.variant=="fftrep")&(S400.P==1)].iloc[0].t
for a,y in zip(ax,[Te,t1/Te,t1/Te/Pe]): a.plot(Pe,y,"D",mfc="white",mec=NEU["k"],mew=1.6,ms=6,ls="none",zorder=6)
Pm=np.array([1,2,4,8,16,32,64,128,256,512]); ax[1].plot(Pm,Pm,":",color=C["gray"],label="ideal")
ax[0].set_ylabel("time per iteration (s)"); ax[0].set_title("(a) Time per iteration")
ax[1].set_ylabel("speed-up vs. 1 process"); ax[1].set_title("(b) Speed-up saturates at 29-43x")
ax[2].axhline(1,color=C["gray"],ls=":"); ax[2].set_ylabel("efficiency = speed-up / P"); ax[2].set_title("(c) Efficiency: 0.05-0.08 at 512")
for a in ax[:2]: a.set_yscale("log")
for a in ax: a.set_xlabel("MPI processes P"); pticks(a,[1,8,64,512]); a.tick_params(axis="x",labelsize=11.5)
ax[2].set_ylim(0,1.08); ax[2].set_xlim(0.8,1400)
from matplotlib.lines import Line2D
hs=[Line2D([],[],color=c,marker="o",ms=5,lw=2) for _,_,c,_ in series]; ls=[f"{l}: peak {df[df.variant=='fftdist'].s.max():.1f}x" for l,df,_,_ in series]
hs+=[Line2D([],[],color=C["gray"],lw=2),Line2D([],[],color=C["gray"],lw=2,ls="--"),Line2D([],[],color=C["gray"],lw=1.5,ls=":"),Line2D([],[],color=NEU["k"],marker="D",mfc="white",mew=1.6,ms=6,ls="none")]; ls+=["distributed FFT","replicated FFT","ideal","evenly filled nodes (400 boxes)"]
fig.legend(hs,ls,loc="lower center",ncol=2,fontsize=11.5,frameon=False,bbox_to_anchor=(0.5,-0.01))
shrink(fig,"R2_strong_scaling"); fig.tight_layout(rect=(0,0.25,1,1)); fig.savefig(f"{OUT}/R2_strong_scaling.png"); plt.close(fig); print("wrote R2_strong_scaling"); plt.rcParams.update({"font.size":10.5,"axes.titlesize":11.5,"axes.labelsize":10.5,"xtick.labelsize":10.5,"ytick.labelsize":10.5,"legend.fontsize":10})

In [ ]:
# ---- R3 stage stacked bars ----------------------------------------------------------------------------------------------
s=pd.read_csv(f"{TAB}/fig2_stage_breakdown.csv"); P=s.P.astype(int).tolist(); x=np.arange(len(P))
st_=[("steady_fft","FFT compute",STG["fft"]),("steady_grid_comm","grid Reduce_scatter",STG["gsum"]),("steady_fft_transpose","two FFT transposes",STG["trans"]),("steady_pot_allgather","potential Allgatherv",STG["allg"]),("steady_exch_comm","coordinate exchange",STG["exch"])]
fig,ax=plt.subplots(figsize=(9.5,4.8)); bot=np.zeros(len(P))
for k,lab,col in st_:
    v=s[k].values; ax.bar(x,v,bottom=bot,color=col,label=lab,width=0.62); bot+=v
rest=np.clip(s.steady_iter.values-bot,0,None); ax.bar(x,rest,bottom=bot,color=STG["update"],label="attraction, spread, gather, update, waiting",width=0.62); bot=bot+rest
ax.plot(x,s.steady_iter.values,"D",color=C["dark"],ms=6,label="measured whole iteration",zorder=5)
for i in range(len(P)):
    comm=s.steady_grid_comm[i]+s.steady_fft_transpose[i]+s.steady_pot_allgather[i]; ax.text(i,max(bot[i],s.steady_iter[i])+14,f"grid comm.\n{comm:.0f} ms",ha="center",fontsize=8.5,color=C["dark"])
ax.set_xticks(x); ax.set_xticklabels(P); ax.set_xlabel("MPI processes P"); ax.set_ylabel("ms per iteration (slowest rank)"); ax.set_ylim(0,max(bot.max(),s.steady_iter.max())*1.18)
ax.set_title("FFT compute shrinks with P, grid communication does not")
ax.legend(fontsize=10,loc="upper center",ncol=3,bbox_to_anchor=(0.5,-0.14)); save(fig,"R3_cpu_stage_bars")

In [ ]:
# ---- R4 placement ------------------------------------------------------------------------------------------------------------
p1=pd.read_csv(f"{CPU}/results_place/place.csv"); p1=p1[p1.P==64]; p2=pd.concat([pd.read_csv(f"{CPU}/results_place256/place.csv"),pd.read_csv(f"{CPU}/results_place8/place.csv")])
pl=pd.concat([p1,p2]).groupby(["P","K","nodes"]).median(numeric_only=True).reset_index().sort_values(["P","K"],ascending=[True,False])
lbl=[f"P={int(r.P)}\n{int(r.nodes)} node{'s' if r.nodes>1 else ''}\n"+("3x72 + 40" if (r.P==256 and r.K==72) else f"{int(r.K)}/node") for r in pl.itertuples()]
plt.rcParams.update({"font.size":12,"axes.labelsize":12,"axes.titlesize":12.5,"ytick.labelsize":11,"legend.fontsize":11}); fig,ax=plt.subplots(figsize=(10.6,4.6)); x=np.arange(len(pl)); bot=np.zeros(len(pl))
for k,lab,col in [("fft","FFT compute",STG["fft"]),("grid_comm","grid Reduce_scatter",STG["gsum"]),("fft_transpose","two FFT transposes",STG["trans"]),("pot_allgather","potential Allgatherv",STG["allg"])]:
    v=pl[k].values*1000; ax.bar(x,v,bottom=bot,color=col,label=lab,width=0.6); bot+=v
ax.plot(x,pl.sec_per_iter_steady.values*1000,"D",color=C["dark"],ms=6,label="measured whole iteration",zorder=5)
for i,t in enumerate(pl.sec_per_iter_steady.values*1000): ax.text(i,max(bot[i],t)+17,f"{t:.0f} ms",ha="center",fontsize=10.5,fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(lbl,fontsize=10.5); ax.set_ylabel("ms per iteration"); ax.set_ylim(0,max(bot.max(),pl.sec_per_iter_steady.max()*1000)*1.5)
ax.set_title("Same P on more nodes: full nodes cost about 2%, the uneven fill at P=256 about 23%"); ax.legend(fontsize=10.5,loc="upper left",ncol=2); save(fig,"R4_placement_bars"); plt.rcParams.update({"font.size":10.5,"axes.titlesize":11.5,"axes.labelsize":10.5,"xtick.labelsize":10.5,"ytick.labelsize":10.5,"legend.fontsize":10})

In [ ]:
# ---- R5 exchange variants -----------------------------------------------------------------------------------------------------
f=pd.read_csv(f"{CPU}/results_vcpu_F/F_summary.csv"); f=f[f.data=="g250k"]; Ps=[16,64,128]
fr=pd.read_csv(f"{CPU}/results_vcpu_F_rep/F_summary.csv"); fr=fr[fr.data=="g250k"]   # P=128 again, 5 repeats, separate job
nice={"F1_allgather_block":"A1","F2_allgather_nonblock":"A2","F3_ghost_block":"A3","F4_ghost_nonblock":"A4","F4poll_ghost_nonblock":"A4\npolling"}
fig,ax=plt.subplots(1,3,figsize=(13.5,4),sharey=False)
for a,p in zip(ax,Ps):
    src=fr if p==128 else f; vs=set(src.variant); d=src[src.P==p].set_index("variant").reindex([v for v in nice if v in vs]); cols=[NEU["lg"],NEU["g"],NEU["br"],NEU["k"],NEU["k"]]
    bars=a.bar(range(len(d)),d.iter_ms,color=cols[:len(d)],width=0.62)
    if len(d)==5: bars[4].set_hatch("///"); bars[4].set_edgecolor("white")
    for i,(t,sh) in enumerate(zip(d.iter_ms,d.exch_share)): a.text(i,t*1.01,f"{t:.0f} ms\n{sh*100:.0f}%",ha="center",fontsize=9.5)
    a.set_xticks(range(len(d))); a.set_xticklabels([nice[v] for v in d.index],fontsize=8); a.set_ylim(0,d.iter_ms.max()*1.22); a.set_title(f"P={p}"+(" (5 repeats)" if p==128 else "")+f": spread {100*(d.iter_ms.max()/d.iter_ms.min()-1):.1f}%"); a.grid(axis="x",visible=False)
ax[0].set_ylabel("ms per iteration")
fig.suptitle("Coordinate-exchange method changes the total time by less than 5%",fontweight="bold",fontsize=12); save(fig,"R5_exchange_variants")

In [ ]:
# ---- R6 weak scaling -------------------------------------------------------------------------------------------------------------
w=pd.read_csv(f"{TAB}/weak256_summary.csv"); lab={"fixed_B1":("fixed 128 boxes",NEU["br"],"^-"),"sqrtgrid_fftdist":("grid area proportional to N",NEU["k"],"o-")}
plt.rcParams.update({"font.size":13,"axes.titlesize":13,"axes.labelsize":12.5,"xtick.labelsize":11.5,"ytick.labelsize":11.5,"legend.fontsize":10.5})
fig,ax=plt.subplots(1,2,figsize=(12.5,5.0))
for k,(l,col,m) in lab.items():
    d=w[w.variant==k]; ax[0].plot(d.P,d.sec_per_iter_mean,m,color=col,label=l); ax[1].plot(d.P,d.weak_efficiency,m,color=col,label=l)
    e=d[d.P==256]; ax[1].annotate(f"{e.weak_efficiency.iloc[0]:.2f}" if e.weak_efficiency.iloc[0]>0.1 else f"{e.weak_efficiency.iloc[0]:.3f}",(256,e.weak_efficiency.iloc[0]),xytext=((-8,8) if e.weak_efficiency.iloc[0]>0.1 else (-10,-13)),textcoords="offset points",color=col,fontweight="bold",ha="right")
j5=json.load(open(f"{CPU}/results_vcpu_weak512/Bweak/sqrtgrid_fftdist_P512_g2560000_r1/summary.json")); t5=j5["seconds_per_iter_steady"]; e5=float(w[(w.variant=="sqrtgrid_fftdist")&(w.P==1)].sec_per_iter_mean.iloc[0])/t5
ax[0].plot([512],[t5],"o",mfc="white",color=NEU["k"],ms=8); ax[1].plot([512],[e5],"o",mfc="white",color=NEU["k"],ms=8); ax[1].annotate(f"{e5:.3f}\n(1 round)",(512,e5),xytext=(0,24),textcoords="offset points",color=NEU["k"],fontweight="bold",ha="center",fontsize=10.5)
Pw=[1,16,64,128,256,512]; ax[0].set_yscale("log"); ax[0].set_ylabel("time per iteration (s)"); ax[0].set_title("(a) 5000 points per rank: ideal is a flat line"); ax[0].legend(fontsize=10.5,loc="center left",bbox_to_anchor=(0.02,0.6))
ax[1].axhline(1,color=C["gray"],ls=":"); ax[1].set_ylim(0,1.08); ax[1].set_ylabel("weak-scaling efficiency T(1)/T(P)"); ax[1].set_title("(b) Efficiency (hollow: P=512, one round)")
for a in ax: a.set_xlabel("MPI processes P"); pticks(a,Pw)
save(fig,"R6_weak_scaling"); plt.rcParams.update({"font.size":10.5,"axes.titlesize":11.5,"axes.labelsize":10.5,"xtick.labelsize":10.5,"ytick.labelsize":10.5,"legend.fontsize":10})

In [ ]:
# ---- R19 where the time of the growing-grid weak-scaling runs goes (Table of stages as a figure) -------------------------------------------------------
try:
    ws=pd.read_csv(next(f for f in (f"{TAB}/weak_stages.csv","work/source/weak_stages.csv") if os.path.exists(f)))
    rest=ws.iter_ms-ws.fft_ms-ws.grid_stages_ms
    comp=[("FFT compute",STG["fft"],ws.fft_ms),("grid sum (Reduce_scatter)",STG["gsum"],ws.reduce_scatter_ms),("transposes",STG["trans"],ws.transposes_ms),("Allgatherv",STG["allg"],ws.allgatherv_ms),("rest (spread, attraction, update)",STG["update"],rest)]
    fig,ax=plt.subplots(figsize=(7.6,4.4)); bot=np.zeros(len(ws)); x=np.arange(len(ws))
    for l,col,v in comp:
        v=np.asarray(v,float); ax.bar(x,v,bottom=bot,color=col,label=l,width=0.58,edgecolor="white",linewidth=1.2); bot+=v
    for i,(t,sh) in enumerate(zip(ws.iter_ms,ws.grid_share)): ax.text(i,t+6,f"{t:.0f} ms\ngrid stages {100*sh:.0f}%",ha="center",va="bottom",fontsize=9.5,color="#222222")
    ax.set_xticks(x); ax.set_xticklabels([f"P={p}" for p in ws.P]); ax.set_ylabel("ms per iteration"); ax.set_ylim(0,ws.iter_ms.max()*1.2); ax.grid(axis="x",visible=False)
    h,l=ax.get_legend_handles_labels(); ax.legend(h[::-1],l[::-1],loc="upper left",fontsize=9.5)
    save(fig,"R19_weak_stages")
except Exception as e: print("skip R19:",e)

In [ ]:
# ---- R7 GPU stage bars (1 GPU vs 2 GPUs) -------------------------------------------------------------------------------------------------------
def js(p): return json.load(open(p))
runs=[("1 GPU",js(f"{GPU}/results_fx/base/base_P1_g1m_r1/summary.json")),("2 GPUs,\nCUDA-aware (device)",js(f"{GPU}/results_fx_device/A/A1_allgather_block_P2_g1m_r1/summary.json")),("2 GPUs,\nhost-staged",js(f"{GPU}/results_fx_hpcxhost/A/A1_allgather_block_P2_g1m_r1/summary.json"))]
comp=[("near","sparse attraction",STG["attr"],lambda j:j["steady_near_local"]+j["steady_near_remote"]),("fft","FFT convolution",STG["fft"],lambda j:j["steady_fft"]),("ks","FFT of the kernel",STG["kernel"],lambda j:j["steady_kernel_spec"]),
      ("gc","grid-sum communication",STG["gsum"],lambda j:j["steady_grid_comm"]),("ex","coordinate exchange",STG["exch"],lambda j:j["steady_exch_comm"])]
fig,ax=plt.subplots(figsize=(8.2,4.8)); bot=np.zeros(3)
for k,l,col,fn in comp:
    v=np.array([fn(j)*1000 for _,j in runs]); ax.bar(range(3),v,bottom=bot,color=col,label=l,width=0.55); bot+=v
tot=np.array([j["seconds_per_iter_steady"]*1000 for _,j in runs]); rest=tot-bot; ax.bar(range(3),rest,bottom=bot,color="#cfcdc7",label="spread, gather, update, other",width=0.55)
for i,t in enumerate(tot): ax.text(i,t+3,f"{t:.0f} ms"+("" if i==0 else f"  ({tot[0]/t:.2f}x)"),ha="center",fontweight="bold",fontsize=10)
ax.set_xticks(range(3)); ax.set_xticklabels([n for n,_ in runs]); ax.set_ylabel("ms per iteration (1M points, fixed grid)"); ax.set_ylim(0,tot.max()*1.15); ax.grid(axis="x",visible=False)
ax.set_title("A second GPU shortens the attraction by 40% but not the FFT,\nand adds grid-sum communication"); ax.legend(fontsize=10,loc="upper left"); save(fig,"R7_gpu_stage_bars")

In [ ]:
# ---- R8 GPU transport -----------------------------------------------------------------------------------------------------------------------------
def gl(d):
    v=[]
    for sub in ["A","B"]:
        for p in glob.glob(f"{GPU}/{d}/{sub}/*P2_g1m_r*/summary.json"):
            j=js(p); v.append(j.get("loop_seconds_excl_kl",j["loop_seconds"])/j["iters"]*1000)
    return st.median(v)
tr=[("same node\nhost-staged","results_fx_hpcxhost",NEU["lg"]),("same node\nCUDA-aware","results_fx_device",NEU["k"]),("two nodes\nhost-staged","results_fx_hpcxhost_2n_u",NEU["lg"]),("two nodes\nCUDA-aware","results_fx_device_2n_u",NEU["k"])]
vals=[gl(d) for _,d,_ in tr]; one=runs[0][1]["seconds_per_iter_steady"]*1000
fig,ax=plt.subplots(figsize=(8,4.6)); ax.bar(range(4),vals,color=[c for _,_,c in tr],width=0.6); ax.axhline(one,color=C["red"],ls="--",lw=1.2,label=f"one GPU: {one:.1f} ms"); ax.legend(loc="upper center",fontsize=9.5)
for i,v in enumerate(vals): ax.text(i,v+3,f"{v:.0f} ms ({one/v:.2f}x)",ha="center",fontweight="bold",fontsize=9.5)
ax.set_xticks(range(4)); ax.set_xticklabels([n for n,_,_ in tr]); ax.set_ylim(0,max(vals)*1.18); ax.set_ylabel("ms per iteration (2 GPUs, 1M points)"); ax.grid(axis="x",visible=False)
ax.set_title("CUDA-aware MPI is faster within a node, slower across nodes\n(speed-up in brackets is relative to one GPU)"); save(fig,"R8_gpu_transport")

In [ ]:
# ---- R9 official comparison ------------------------------------------------------------------------------------------------------------------------------
o=pd.read_csv(f"{TAB}/official_compare_summary.csv"); fig,ax=plt.subplots(1,2,figsize=(12,4.6),sharey=True); cores=[1,8,36,72]
for a,d in zip(ax,["g50k","ts50k"]):
    off=o[(o.data==d)&(o.impl=="official")].set_index("threads_or_P").loop_s_mean; ours=pd.concat([o[(o.data==d)&(o.impl=="ours_serial")].set_index("threads_or_P").loop_s_mean,o[(o.data==d)&(o.impl=="ours_mpi")].set_index("threads_or_P").loop_s_mean])
    x=np.arange(4); a.bar(x-0.19,[off[c] for c in cores],0.36,color=NEU["lg"],label="official FIt-SNE (OpenMP threads, FFTW)"); a.bar(x+0.19,[ours[c] for c in cores],0.36,color=NEU["k"],label="ours (serial / MPI ranks)")
    for i,c in enumerate(cores):
        a.text(i-0.19,off[c]*1.05,f"{off[c]:.0f}",ha="center",fontsize=8.5); a.text(i+0.19,ours[c]*1.05,f"{ours[c]:.0f}",ha="center",fontsize=8.5,color=NEU["k"]); r_=off[c]/ours[c]; a.text(i,2200,(f"{r_:.1f}\u00d7\nfaster" if r_>=1 else f"{1/r_:.1f}\u00d7\nslower"),ha="center",va="center",fontsize=9.5,fontweight="bold",color=NEU["k"])
    a.set_yscale("log"); a.set_ylim(3,30000); a.set_xticks(x); a.set_xticklabels(cores); a.set_xlabel("cores (threads or MPI ranks)"); a.set_title(d); a.grid(axis="x",visible=False)
ax[0].set_ylabel("loop time for 1000 iterations (s)"); ax[0].legend(fontsize=10,loc="upper right",title="bold factor: ours relative to official",title_fontsize=9.5)
fig.suptitle("Ours is 2.7-3.4x slower on one core and 4.2-4.5x faster on 72 cores than the official code",fontweight="bold",fontsize=12); save(fig,"R9_official_compare")

In [ ]:
# ---- R0 pipeline ----------------------------------------------------------------------------------------------------------------------------------------
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(13,4.2)); ax.axis("off"); ax.set_xlim(0,13); ax.set_ylim(0,4.2); ax.grid(False)
def box(x,y,w,t,col="#dbe8fb",ec=C["blue"]):
    ax.add_patch(FancyBboxPatch((x,y),w,0.9,boxstyle="round,pad=0.03",fc=col,ec=ec,lw=1.4)); ax.text(x+w/2,y+0.45,t,ha="center",va="center",fontsize=9.5)
def arr(x1,x2,y,col=C["gray"],lab=None):
    ax.annotate("",xy=(x2,y),xytext=(x1,y),arrowprops=dict(arrowstyle="-|>",color=col,lw=1.6))
    if lab: ax.text((x1+x2)/2,y+0.12,lab,ha="center",fontsize=8,color=col)
stg=["spread\n(own points)","kernel\nspectrum","FFT\nconvolution","gather\n(own points)","attraction\n+ update"]
xs=[0.2,2.9,5.3,8.0,10.7]; ws=[2.1,1.9,2.2,2.1,2.1]
ax.text(0.1,3.85,"Serial iteration: all stages on one core",fontsize=10.5,fontweight="bold")
for x,w,t in zip(xs,ws,stg): box(x,2.75,w,t)
for i in range(4): arr(xs[i]+ws[i]+0.03,xs[i+1]-0.03,3.2)
ax.text(0.1,2.2,"Parallel iteration: stages added by MPI (orange: does not shrink with P)",fontsize=10.5,fontweight="bold")
pst=[("coordinate\nexchange",C["gray"]),("grid sum\nReduce_scatter",C["orange"]),("row FFT, transpose,\ncolumn FFT (x2)",C["orange"]),("potential\nAllgatherv",C["orange"]),("Allreduce\nof scalars",C["gray"])]
xs2=[0.2,2.9,5.3,8.0,10.7]
for (t,col),x,w in zip(pst,xs2,ws): box(x,0.9,w,t,col="#ffffff" if col==C["gray"] else "#fde3d7",ec=col)
for i in range(4): arr(xs2[i]+ws[i]+0.03,xs2[i+1]-0.03,1.35)
ax.text(0.2,0.45,"before attraction",fontsize=8.5,color=C["gray"]); ax.text(2.9,0.45,"spread -> grid sum",fontsize=8.5,color=C["orange"]); ax.text(5.3,0.45,"FFT compute / P, transposes ~ grid",fontsize=8.5,color=C["orange"]); ax.text(8.0,0.45,"before gather",fontsize=8.5,color=C["orange"]); ax.text(10.7,0.45,"Z, bounding box, centering",fontsize=8.5,color=C["gray"])
save(fig,"R0_pipeline")

In [ ]:
# ---- R10 CPU best vs GPU double/float (same grid) --------------------------------------------------------------------------------------------------
sg=pd.read_csv(f"{GPU}/results_gpu_samegrid/samegrid.csv").groupby(["data","boxes","precision"]).sec_per_iter_steady.median()*1000
cpu={("g250k",400):S400.t.min()*1000,("g250k",512):S512.t.min()*1000,("g1m",768):S768.t.min()*1000}
fig,ax=plt.subplots(figsize=(9,4.6)); keys=list(cpu); x=np.arange(3); w=0.26
ax.bar(x-w,[cpu[k] for k in keys],w,color=NEU["lg"],label="best CPU run (distributed FFT, double)"); ax.bar(x,[sg[(*k,"double")] for k in keys],w,color=NEU["k"],label="1 GPU, double"); ax.bar(x+w,[sg[(*k,"float")] for k in keys],w,color=NEU["br"],label="1 GPU, single")
for i,k in enumerate(keys):
    c=cpu[k]; d=sg[(*k,"double")]; f=sg[(*k,"float")]
    ax.text(i-w,c*1.08,f"{c:.0f}",ha="center",fontsize=8.5); ax.text(i,d*1.08,f"{d:.0f}\n({c/d:.1f}x)",ha="center",fontsize=8.5,color=NEU["k"]); ax.text(i+w,f*1.08,f"{f:.0f}\n({c/f:.1f}x)",ha="center",fontsize=8.5,color=NEU["br"])
ax.set_yscale("log"); ax.set_ylim(8,3000); ax.set_xticks(x); ax.set_xticklabels(["250k points,\n400 boxes","250k points,\n512 boxes","1M points,\n768 boxes"]); ax.set_ylabel("ms per iteration (log)"); ax.grid(axis="x",visible=False); ax.legend(fontsize=10,loc="upper left")
ax.set_title("On identical grids one GPU is 7-10x faster than the best CPU run (up to 512 cores)"); save(fig,"R10_gpu_vs_cpu")

In [ ]:
# ---- R11 full-run crossover ------------------------------------------------------------------------------------------------------------------------------
c=pd.read_csv(f"{CPU}/results_vanda_cross/crossover.csv").groupby(["N","method"]).median(numeric_only=True).reset_index()
e=c[c.method=="exact"]; f=c[c.method=="fast"]
fig,ax=plt.subplots(1,2,figsize=(11.5,4.4))
ax[0].plot(e.N,e.loop_seconds,"o-",color=NEU["br"],label="exact t-SNE"); ax[0].plot(f.N,f.loop_seconds,"s-",color=NEU["k"],label="FIt-SNE (adaptive grid)")
Nx=np.array([2000,20000]); ax[0].plot(Nx,e.loop_seconds.iloc[-1]*(Nx/10000)**2,":",color=C["gray"],label=r"$N^2$ extrapolation")
ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("number of points N"); ax[0].set_ylabel("time for 1000 iterations (s)"); ax[0].legend(fontsize=10); ax[0].axvline(1.5e4,color=C["gray"],ls="-.",lw=1); ax[0].text(1.5e4*1.04,60,"extrapolated\nbreak-even\n1.5e4",fontsize=10,color=C["gray"]); ax[0].set_xlim(1700,2.4e4); ax[0].set_xticks([2000,5000,10000,20000]); ax[0].set_xticklabels(["2e3","5e3","1e4","2e4"]); ax[0].minorticks_off(); ax[0].set_title("(a) Whole runs of 1000 iterations")
for col,lab,cl in [("med_it11_250","iterations 11-250 (early exaggeration)","#9a9893"),("med_it251_600","iterations 251-600","#555450"),("med_it601_end","iterations 601-1000",NEU["k"])]: ax[1].plot(f.N,f[col],"o-",color=cl,label=lab)
ax[1].plot(e.N,e.sec_per_iter_mean,"o--",color=NEU["br"],label="exact t-SNE (any phase)")
ax[1].set_yscale("log"); ax[1].set_xscale("log"); ax[1].set_xlabel("number of points N"); ax[1].set_ylabel("time per iteration (s)"); ax[1].set_ylim(top=40); ax[1].legend(fontsize=10,loc="upper left",ncol=1,handlelength=1.6); ax[1].set_title("(b) FIt-SNE iteration cost grows during the run")
save(fig,"R11_crossover")

In [ ]:
# ---- R12 KL gap -----------------------------------------------------------------------------------------------------------------------------------------------
k=pd.read_csv(f"{CPU}/results_klgap/klgap.csv").groupby(["data","variant"]).kl_common.mean().reset_index()
fig,ax=plt.subplots(1,2,figsize=(12,4.4))
for a,d,vs in zip(ax,["g10000","ts10k"],[["exact","official","ours","ours_bw05","ours_p5","ours_long"],["official","ours","ours_bw05"]]):
    base=k[(k.data==d)&(k.variant==("exact" if d=="g10000" else "official"))].kl_common.iloc[0]; v=[k[(k.data==d)&(k.variant==x)].kl_common.iloc[0] for x in vs]
    nm={"exact":"exact\nt-SNE","official":"official\nFIt-SNE","ours":"ours,\nbox width 1","ours_bw05":"ours,\nbox width 0.5","ours_p5":"ours, box 1,\np=5","ours_long":"ours, box 1,\n2000 iters"}
    cols=[C["gray"] if x in("exact","official") else C["blue"] for x in vs]; a.axhline(base,color=C["gray"],ls="--",lw=1.1)
    a.scatter(range(len(v)),v,c=cols,s=70,zorder=3)
    for i,y in enumerate(v): a.text(i,y+(max(v)-min(v))*0.07+0.0004,f"{y:.3f}"+("" if abs(y-base)<1e-12 else f"\n({100*(y/base-1):+.1f}%)"),ha="center",va="bottom",fontsize=8.5)
    lo,hi=min(v),max(v); pad=(hi-lo)*0.25+0.003; a.set_ylim(lo-pad,hi+pad*2.2); a.set_xlim(-0.5,len(v)-0.5); a.set_xticks(range(len(v))); a.set_xticklabels([nm[x] for x in vs],fontsize=8.5); a.set_ylabel("KL divergence (axis does not start at 0)"); a.set_title({"g10000":"g10000 (10k synthetic points)","ts10k":"ts10k (10k cells)"}[d]); a.grid(axis="x",visible=False)
fig.suptitle("The KL gap to the official code is a grid-resolution effect: a finer grid (width 0.5) closes it",fontweight="bold",fontsize=12); save(fig,"R12_kl_gap")

In [ ]:
# ---- R13 ablations X1/X2 ------------------------------------------------------------------------------------------------------------------------------------
a=pd.read_csv(f"{CPU}/results_ablation/ablation.csv").groupby(["P","config"]).sec_per_iter_steady.median().unstack()
rel=lambda c:(a[c]/a["base_pc1_ghost"]-1)*100
fig,ax=plt.subplots(1,2,figsize=(11.5,4.4)); x=np.arange(3); w=0.26
ax[0].bar(x-w,rel("random_ghost"),w,color=NEU["br"],label="random blocks, ghost exchange"); ax[0].bar(x,rel("random_allgather"),w,color=NEU["lg"],label="random blocks, allgather"); ax[0].bar(x+w,rel("pc1_allgather"),w,color=NEU["k"],label="PC1 blocks, allgather")
for i,c in enumerate(["random_ghost","random_allgather","pc1_allgather"]):
    for j,v in enumerate(rel(c).values): ax[0].text(j+(i-1)*w,max(v,0)+0.15,f"{v:+.1f}%",ha="center",fontsize=8.5)
ax[0].axhline(0,color=C["dark"],lw=0.8); ax[0].set_xticks(x); ax[0].set_xticklabels([f"P={p}" for p in a.index]); ax[0].set_ylabel("iteration time relative to PC1 + ghost (%)"); ax[0].legend(fontsize=10,loc="upper left"); ax[0].grid(axis="x",visible=False)
ax[0].set_title("(a) Random blocks are 2-5% slower with the ghost exchange")
ax[1].bar(x,rel("gridcomm_nonblocking"),0.5,color=NEU["g"])
for j,v in enumerate(rel("gridcomm_nonblocking").values): ax[1].text(j,v+0.3,f"{v:+.1f}%",ha="center",fontsize=9,fontweight="bold")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"P={p}" for p in a.index]); ax[1].set_ylabel("iteration time relative to blocking grid sum (%)"); ax[1].set_title("(b) Non-blocking grid sum is slower, not faster"); ax[1].grid(axis="x",visible=False)
save(fig,"R13_ablation_order_gridcomm")

In [ ]:
# ---- R14 interpolation order: error vs cost ------------------------------------------------------------------------------------------------------------------------
pc=pd.read_csv(f"{CPU}/results_pcost/pcost.csv").groupby("p").total.median(); ge=pd.read_csv(next(f for f in (f"{CPU}/exact_validation_vanda/grad_errors.csv",f"{CPU}/results_validation/exact_validation/grad_errors.csv") if os.path.exists(f)))
err=[ge[(ge.coords=="it50")&(ge.candidate==("fast_fixed128" if q==3 else f"fast_fixed128_p{q}"))].rel_grad.iloc[0] for q in [2,3,4,5]]
fig,ax=plt.subplots(1,2,figsize=(10.5,4.2)); ps=[2,3,4,5]
ax[0].semilogy(ps,err,"o-",color=NEU["k"]); ax[0].set_xticks(ps); ax[0].set_xlabel("interpolation nodes per box p"); ax[0].set_ylabel("relative gradient error (128 boxes, iteration 50)"); ax[0].set_title("(a) Each extra node gains ~2 orders of magnitude")
for q,e_ in zip(ps,err): ax[0].annotate(f"{e_:.1e}",(q,e_),xytext=(4,6),textcoords="offset points",fontsize=8.5)
ax[1].bar(ps,pc.values,color=NEU["lg"],width=0.6); ax[1].plot(ps,pc.values[0]*np.array(ps)**2/4,"k:",label=r"$\propto p^2$"); ax[1].set_xticks(ps); ax[1].set_xlabel("interpolation nodes per box p"); ax[1].set_ylabel("serial time per iteration (s)")
for q,t in zip(ps,pc.values): ax[1].text(q,t+0.08,f"{t:.2f} s",ha="center",fontsize=9)
ax[1].legend(fontsize=10); ax[1].set_title("(b) Cost grows as p^2 (grid nodes = (p n_b)^2)"); ax[1].grid(axis="x",visible=False)
save(fig,"R14_interp_order")

In [ ]:
# ---- R15/R16 real data: embedding and per-type purity ------------------------------------------------------------------------------------------------------------
RD=A.data; emb=f"{GPU}/results_fx_real/real/real_P1_ts_r1/embedding.bin"
Y=np.fromfile(emb,dtype=np.float64,offset=16).reshape(-1,2); lab=np.fromfile(f"{RD}/ts.labels.bin",dtype=np.int32,offset=16); names=[l.strip() for l in open(f"{RD}/ts.label_names.txt")]
assert len(Y)==len(lab),(len(Y),len(lab))
cnt=np.bincount(lab,minlength=len(names)); top=np.argsort(-cnt)[:12]
def short(n):
    for a_,b_ in [("CD4-positive, alpha-beta","CD4+ \u03b1\u03b2"),("CD8-positive, alpha-beta","CD8+ \u03b1\u03b2"),("luminal epithelial cell of mammary gland","luminal mammary epithelial cell"),("mesenchymal stem cell","mesenchymal stem cell")]: n=n.replace(a_,b_)
    return n
rng=np.random.default_rng(0); Yf=Y.astype(np.float32)
def purity(q,k=15):   # fraction of the k nearest embedding neighbours (among all N points) that share the label
    out=np.empty(len(q)); sq=(Yf**2).sum(1)
    for i in range(0,len(q),40):
        b=q[i:i+40]; d=sq[None,:]-2*Yf[b]@Yf.T; d[np.arange(len(b)),b]=np.inf
        nn=np.argpartition(d,k,axis=1)[:,:k]; out[i:i+40]=(lab[nn]==lab[b][:,None]).mean(1)
    return out
q_all=rng.choice(len(Y),6000,replace=False); overall=purity(q_all).mean(); print("overall purity (6000 random cells)",overall)
q_typ=np.concatenate([rng.choice(np.where(lab==t)[0],min(120,cnt[t]),replace=False) for t in range(len(names)) if cnt[t]>0]); pq=purity(q_typ)
pur=np.full(len(names),np.nan)
for t in range(len(names)):
    m=lab[q_typ]==t
    if m.any(): pur[t]=pq[m].mean()
fig,ax=plt.subplots(figsize=(8.5,7.2)); ax.scatter(Y[:,0],Y[:,1],s=0.3,c="#d9d8d3",rasterized=True)
COL12=["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#bcbd22","#17becf","#111111","#e6ab02","#7a7a00"]
COL12[11]="#4d4dff"
from matplotlib.lines import Line2D
hs=[]
for i,t in enumerate(top):
    m=lab==t; ax.scatter(Y[m,0],Y[m,1],s=0.5,color=COL12[i],rasterized=True); hs.append(Line2D([],[],marker="o",ls="",color=COL12[i],ms=7,label=f"{short(names[t])} ({cnt[t]/1000:.0f}k)"))
hs.append(Line2D([],[],marker="o",ls="",color="#d9d8d3",ms=7,label="all other types"))
ax.set_xticks([]); ax.set_yticks([]); ax.grid(False); ax.set_title("Tabula Sapiens, 483k cells: the 12 largest cell types")
ax.legend(handles=hs,loc="upper center",bbox_to_anchor=(0.5,-0.01),ncol=3,fontsize=8,columnspacing=1.0,handletextpad=0.2,borderaxespad=0.2)
save(fig,"R15_real_embedding")
fig,ax=plt.subplots(figsize=(8.5,5)); ok=cnt>0
ax.scatter(cnt[ok],pur[ok],s=22,color=C["blue"],alpha=0.8); ax.set_xscale("log"); ax.set_xlabel("number of cells of the type"); ax.set_ylabel("label purity of the type (15 neighbors)"); ax.set_ylim(0,1.14)
idx=np.where(cnt>=1000)[0]; srt=idx[np.argsort(pur[idx])]
lo3=list(srt[:3]); hi2=sorted(srt[-2:],key=lambda t:cnt[t])
for t,off in zip(lo3+hi2,[(-12,-16),(14,-16),(14,-16),(-10,18),(10,18)]):   # lowest three: the label is put into the empty area under the point
    ax.annotate(short(names[t]),(cnt[t],pur[t]),xytext=off,textcoords="offset points",fontsize=7.5,ha="left" if off[0]>0 else "right",va="center",arrowprops=dict(arrowstyle="-",color=C["gray"],lw=0.6))
ax.axhline(overall,color=C["gray"],ls=":"); ax.text(cnt[ok].min()*1.1,overall+0.02,f"overall {overall:.2f}",ha="left",fontsize=8.5,color=C["gray"])
ax.set_title("Large cell types separate well; small types and lineage subtypes do not"); save(fig,"R16_type_purity")

In [ ]:
# ---- Fig. 6 real data on the CPU (R1) -------------------------------------------------------------------------------------------------------------------------
try:
    rs=pd.read_csv(f"{TAB}/real_scaling.csv"); a_=rs[rs.job=="a"].set_index("P").sec_per_iter; b_=rs[rs.job=="b"].set_index("P").sec_per_iter; t1=a_.loc[1]
    fig,ax=plt.subplots(1,2,figsize=(11,4.3))
    ax[0].plot(a_.index,a_.values,"o-",color=NEU["k"],label="single node"); ax[0].plot(b_.index,b_.values,"s-",color=NEU["br"],label="2-4 nodes")
    ax[0].set_yscale("log"); ax[0].set_ylabel("time per iteration (s)"); ax[0].set_title("(a) ts50k, 540 boxes fixed"); ax[0].legend(fontsize=10)
    sp=pd.concat([a_,b_[~b_.index.isin(a_.index)]]); ax[1].plot(sp.index,t1/sp.values,"o-",color=NEU["k"]); Pm=np.array([1,4,12,36,72,144,288]); ax[1].plot(Pm,Pm,":",color=C["gray"],label="ideal")
    ax[1].set_yscale("log"); ax[1].set_ylabel("speed-up vs. 1 process"); ax[1].set_title("(b) Speed-up (T(1) from the single-node job)"); ax[1].legend(fontsize=10)
    for a in ax: a.set_xlabel("MPI processes P"); a.set_xscale("log"); a.set_xticks([1,4,12,36,72,144,288]); a.set_xticklabels(["1","4","12","36","72","144","288"]); a.minorticks_off()
    save(fig,"R17_real_cpu_scaling")
except Exception as e: print("skip R17:",e)

In [ ]:
# ---- Fig. 8 cost model T(P)=a/P+b fitted on P<=64 -------------------------------------------------------------------------------------------------------------
try:
    mf=pd.read_csv(f"{TAB}/comm_model_fit.csv"); mf=mf[mf.model=="a/P+b"]; plt.rcParams.update({"font.size":12.5,"axes.titlesize":12.5,"axes.labelsize":12.5,"xtick.labelsize":11.5,"ytick.labelsize":11.5,"legend.fontsize":11}); fig,ax=plt.subplots(1,3,figsize=(10.4,4.3))
    for a,(name,d,it,_) in zip(ax,[("250k/400 boxes","results_vcpu_E400_big",100,0),("250k/512 boxes","results_vcpu_E512_big",100,0),("1M/768 boxes","results_vcpu_E768_1m",60,0)]):
        e=pd.read_csv(f"{CPU}/{d}/E_summary.csv"); e=e[e.variant=="fftdist"]; r=mf[mf.study==name].iloc[0]
        a.loglog(e.P,e.loop_s_median/it,"o",color=NEU["k"],label="measured"); pp=np.logspace(0,9,100,base=2); a.loglog(pp,r.a/pp+r.b,"-",color=NEU["br"],label=f"fit on P$\\leq$64\nb = {r.b*1e3:.0f} ms")
        a.axvline(64,color="grey",ls=":"); a.set_title(name); a.set_xlabel("MPI processes P"); a.set_ylabel("time per iteration (s)"); a.legend(fontsize=11,loc="upper right")
    save(fig,"R18_cost_model"); plt.rcParams.update({"font.size":10.5,"axes.titlesize":11.5,"axes.labelsize":10.5,"xtick.labelsize":10.5,"ytick.labelsize":10.5,"legend.fontsize":10})
except Exception as e: print("skip R18:",e)

In [ ]:
# ---- Fig. 5 validation against the exact gradient (V1, V2) --------------------------------------------------------------------------------------------------
try:
    gfile = next(f for f in (f"{CPU}/exact_validation_vanda/grad_errors.csv", f"{CPU}/results_validation/exact_validation/grad_errors.csv") if os.path.exists(f))
    ge=pd.read_csv(gfile); ge["boxes"]=ge.candidate.str.extract(r"fixed(\d+)$").astype(float)
    fig,ax=plt.subplots(1,2,figsize=(11,4.4))
    for co,col,l in [("it50",NEU["k"],"it. 50"),("it250",NEU["br"],"it. 250")]:
        d=ge[(ge.coords==co)&ge.boxes.notna()].sort_values("boxes"); ax[0].plot(d.boxes,d.rel_grad,"o-",color=col,label=l+", fixed")
        ad=ge[(ge.coords==co)&(ge.candidate=="fast_adaptive")].rel_grad.iloc[0]; ax[0].axhline(ad,color=col,ls="--",lw=1,label=l+", adaptive")
    ax[0].set_xscale("log",base=2); ax[0].set_yscale("log"); ax[0].set_xlabel("Boxes per dimension"); ax[0].set_ylabel("Relative gradient error vs. exact t-SNE"); ax[0].set_title("(a) FIt-SNE gradient error vs. exact gradient")
    ax[0].set_ylim(top=2e1); ax[0].legend(fontsize=10,loc="upper right",ncol=2,columnspacing=0.8,handlelength=1.6); ax[0].set_xticks([16,32,64,128,256]); ax[0].set_xticklabels([16,32,64,128,256])
    mp=ge[ge.candidate.str.startswith("mpi")&(ge.coords=="it250")]; labs=[re.sub(r"^P(\d+)_(\w+)$",lambda m:f"P={m.group(1)}\n{m.group(2)[:4]}.",c.replace("mpi_","")) for c in mp.candidate]
    ax[1].bar(range(len(mp)),mp.rel_attr*1e16,color=NEU["g"]); ax[1].set_xticks(range(len(mp))); ax[1].set_xticklabels(labs,fontsize=8.5)
    ax[1].set_ylabel("attraction difference vs. serial (x1e-16)"); ax[1].set_title("(b) Parallel vs. serial")
    save(fig,"fig5_exact_validation")
except Exception as e: print("skip fig5:",e)

In [ ]:
# ---- Fig. 11 serial cost versus the number of neighbors K (S1) ------------------------------------------------------------------------------------------------
try:
    f=glob.glob(f"{CPU}/results_vanda_cpu_knn/**/knn.csv",recursive=True)[0]; d=pd.read_csv(f)
    d["K"]=d.name.str.extract(r"_k(\d+)").astype(int); g=d.groupby(["N","K"]).median(numeric_only=True).reset_index()
    fig,ax=plt.subplots(1,2,figsize=(11,4.2)); mk={20000:"o",50000:"^"}
    for n,gg in g.groupby("N"):
        ax[0].plot(gg.K,gg.med_attraction,mk.get(n)+"-",color=STG["attr"],label=f"attraction, N={n//1000}k"); ax[0].plot(gg.K,gg.med_fft,mk.get(n)+"--",color=STG["fft"],label=f"FFT, N={n//1000}k")
        ax[1].plot(gg.K,gg.med_attraction/gg.med_total*100,mk.get(n)+"-",color=STG["attr"],label=f"N={n//1000}k")
    ax[0].set_xscale("log"); ax[0].set_yscale("log"); ax[0].set_xlabel("neighbors per point K (non-zeros = N K)"); ax[0].set_ylabel("time per iteration (s)"); ax[0].set_title("(a) attraction grows with K, FFT does not"); ax[0].legend(fontsize=10)
    ax[1].set_xscale("log"); ax[1].set_xlabel("neighbors per point K"); ax[1].set_ylabel("attraction share of an iteration (%)"); ax[1].set_title("(b) share of the attraction term"); ax[1].legend(fontsize=10)
    for a in ax: a.set_xticks([15,30,60,90,150,300]); a.set_xticklabels([15,30,60,90,150,300]); a.minorticks_off()
    save(fig,"fig11_kscan")
except Exception as e: print("skip fig11:",e)

## 2. Strong scaling with the FFT variants (Figure `q1fft`)

### Figure R_q1fft

Q1-fft figure: speed-up of replicated vs distributed FFT (250k points, 200/400/512 boxes).

In [ ]:
import matplotlib; matplotlib.rcdefaults()   # start from the default style: each figure script sets its own

In [ ]:
import sys, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,"grid.color":"#e4e3df","font.size":11,"legend.frameon":False})
import os
out = FIGS; os.makedirs(out, exist_ok=True)
D = "work/results/s4.3_cpu_strong_scaling"
fig, axs = plt.subplots(1, 3, figsize=(8.2, 3.5), sharey=True)
for ax, d, nb in zip(axs, ("E200","E400","E512"), (200,400,512)):
    t = pd.read_csv(f"{D}/{d}/E_summary.csv"); t1 = t[(t.variant=="fftrep")&(t.P==1)].loop_s_median.iloc[0]
    Ps = sorted(t.P.unique()); ax.plot(Ps, Ps, ":", color="#8a8984", label="ideal")
    for v, lab, col, m in (("fftrep","replicated FFT","#eb6834","s"),("fftdist","distributed FFT","#2a78d6","o")):
        s = t[t.variant==v].sort_values("P"); ax.plot(s.P, t1/s.loop_s_median, m+"-", color=col, label=lab)
    ax.set_xscale("log",base=2); ax.set_yscale("log",base=2); ax.set_xticks(Ps); ax.set_xticklabels([str(p) for p in Ps]); ax.minorticks_off()
    ax.set_yticks([1,2,4,8,16,32]); ax.set_yticklabels(["1","2","4","8","16","32"]); ax.set_title(f"{nb} boxes"); ax.set_xlabel("MPI ranks P")
axs[0].set_ylabel("speed-up over replicated FFT, P=1"); axs[0].legend(fontsize=10)
fig.tight_layout(); fig.savefig(f"{out}/R_q1fft.png", dpi=200); print("ok")

## 3. Distributed FFT on the GPU (Figure R20)

### Figure R20

Figure R20: stages of one iteration of the GPU program with the replicated and the distributed FFT (1M points, 768 boxes).

Reads work/source/gpu_dist_fft_stages.csv   (two GPUs: replicated / distributed FFT, one node and two nodes)
its column dist_1gpu_ms is the distributed FFT on one GPU

In [ ]:
import matplotlib; matplotlib.rcdefaults()   # start from the default style: each figure script sets its own

In [ ]:
import os, sys, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.axisbelow": True, "savefig.dpi": 170, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.7, "font.size": 12, "axes.labelsize": 12, "legend.fontsize": 11, "legend.frameon": False, "axes.edgecolor": "#52514e",
                     "xtick.labelsize": 11, "ytick.labelsize": 11})
STG = dict(fft="#2a78d6", kernel="#6250d6", gsum="#e34948", trans="#e87ba4", allg="#eda100", rest="#cfcdc7")
D = "work/source"
out = FIGS; os.makedirs(out, exist_ok=True)
T = pd.read_csv(f"{D}/gpu_dist_fft_stages.csv").set_index("stage")

In [ ]:
def col(c):   # stage times (ms) of one column of the stage table
    return dict(whole=T.loc["Whole iteration", c],
                fft=T.loc["FFT: row transforms (all of the FFT if replicated)", c] + T.loc["Column and inverse row transforms", c],
                trans=T.loc["Two transposes (copy to the host, Alltoall, copy back)", c], allg=T.loc["Potential Allgatherv", c],
                kernel=T.loc["Kernel spectrum", c], gsum=T.loc["Grid Reduce\\_scatter", c])

In [ ]:
p1 = col("dist_1gpu_ms")   # distributed FFT on one GPU
bars = [("Replicated\nP=2", col("repl_1node_ms")), ("Distributed\nP=1", p1), ("Distributed\nP=2", col("dist_1node_ms")),
        ("Replicated\nP=2", col("repl_2nodes_ms")), ("Distributed\nP=2", col("dist_2nodes_ms"))]
xs = [0, 1, 2, 3.4, 4.4]
parts = [("fft", "FFT", STG["fft"]), ("trans", "two transposes", STG["trans"]), ("allg", "potential Allgatherv", STG["allg"]),
         ("kernel", "kernel spectrum", STG["kernel"]), ("gsum", "grid Reduce_scatter", STG["gsum"])]
fig, ax = plt.subplots(figsize=(7.6, 4.6)); bot = np.zeros(len(bars))
for k, l, c in parts:   # stacked bars, one segment per stage
    v = np.array([b[1][k] for b in bars]); ax.bar(xs, v, bottom=bot, color=c, label=l, width=0.7)
    for x, b, vv in zip(xs, bot, v):
        if vv >= 28: ax.text(x, b + vv / 2, f"{vv:.0f}", ha="center", va="center", fontsize=9.5, color="white" if k in ("fft", "kernel", "gsum") else "#222")
    bot += v
tot = np.array([b[1]["whole"] for b in bars]); rest = np.clip(tot - bot, 0, None)   # everything not listed above
ax.bar(xs, rest, bottom=bot, color=STG["rest"], label="rest", width=0.7)
top = bot + rest
for x, t, tp in zip(xs, tot, top): ax.text(x, tp + 10, f"{t:.0f} ms", ha="center", fontweight="bold", fontsize=10.5)
ax.set_xticks(xs); ax.set_xticklabels([b[0] for b in bars]); ax.set_ylabel("ms per iteration (1M points, 768 boxes)"); ax.set_ylim(0, top.max() * 1.22)
ax.grid(axis="x", visible=False)
ax.annotate("one node", xy=(1, 0), xycoords=("data", "axes fraction"), xytext=(0, -48), textcoords="offset points", ha="center", fontsize=11, fontweight="bold")
ax.annotate("two nodes", xy=(3.9, 0), xycoords=("data", "axes fraction"), xytext=(0, -48), textcoords="offset points", ha="center", fontsize=11, fontweight="bold")
ax.legend(loc="upper left", ncol=3, columnspacing=1.2, handlelength=1.2)
fig.tight_layout(); fig.subplots_adjust(bottom=0.2)
fig.savefig(f"{out}/R20_gpu_distfft.png"); print("ok")

## 4. Data-flow diagram of one iteration (Section 2.3)

### Figure R_dataflow

In [ ]:
import matplotlib; matplotlib.rcdefaults()   # start from the default style: each figure script sets its own

In [ ]:
import sys, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, FancyArrowPatch
STG=dict(fft="#2a78d6",kernel="#6250d6",spread="#1baf7a",attr="#eb6834",update="#8a8984")
INK="#222222"; MUT="#5c5b57"
plt.rcParams.update({"font.family":"DejaVu Sans","axes.unicode_minus":False,"mathtext.default":"regular"})
W,H=11.2,5.7
fig,ax=plt.subplots(figsize=(W,H)); ax.set_xlim(0,W); ax.set_ylim(0,H); ax.axis("off")
def tint(c,a=0.16):
    c=np.array(matplotlib.colors.to_rgb(c)); return tuple(1-(1-c)*a)
def rbox(x,y,w,h,fc,ec,lw=1.6,ls="-",z=3,r=0.1): ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle=f"round,pad=0,rounding_size={r}",fc=fc,ec=ec,lw=lw,ls=ls,zorder=z))
def badge(x,y,col,num,r=0.14):
    ax.add_patch(Circle((x,y),r,fc=col,ec="none",zorder=4)); ax.text(x,y,str(num),color="white",fontsize=8.5,fontweight="bold",ha="center",va="center",zorder=5)
def box(x,y,w,h,col,num,title,sub,sfs=8.8):
    rbox(x,y,w,h,tint(col),col); badge(x+0.22,y+h-0.22,col,num)
    ax.text(x+w/2+0.12,y+h-0.22,title,fontsize=10.5,fontweight="bold",color=INK,ha="center",va="center",zorder=5)
    ax.text(x+w/2,y+(h-0.42)/2,sub,fontsize=sfs,color=INK,ha="center",va="center",zorder=5,linespacing=1.4)
def arrow(p,q,col=MUT,lw=1.6,rad=0): ax.add_patch(FancyArrowPatch(p,q,arrowstyle="-|>",mutation_scale=13,lw=lw,color=col,connectionstyle=f"arc3,rad={rad}",zorder=2,shrinkA=0,shrinkB=0))
def lab(x,y,t,col=MUT,ha="center",fs=8.4): ax.text(x,y,t,fontsize=fs,color=col,ha=ha,va="center",zorder=6,linespacing=1.3)
# bands
rbox(0.08,3.02,W-0.16,2.6,"#eef3fa","none",z=0,r=0.14); rbox(0.08,0.9,W-0.16,1.98,"#f6f3ec","none",z=0,r=0.14)
ax.text(0.25,5.4,"GRID SPACE",fontsize=9.5,fontweight="bold",color=STG["fft"],va="center"); lab(1.55,5.4,"$M=(p\\,n_b)^2$ nodes; the FFT dominates the cost",ha="left",fs=8.8)
ax.text(W-0.25,2.66,"POINT SPACE",fontsize=9.5,fontweight="bold",color="#8c6d1f",va="center",ha="right")
# grid band
GY,GH=3.3,1.6
box(0.35,GY,1.75,GH,STG["update"],1,"Grid","bounding box of $y$\n$n_b\\times n_b$ boxes of width $h$,\n$p^2$ nodes per box")
box(2.7,GY,1.8,GH,STG["spread"],3,"Spread","add the 4 charges of\nevery point to the $p^2$\nnodes of its box")
box(5.1,GY,2.0,GH,STG["fft"],4,"FFT convolution","$\\Phi_c=\\mathcal{F}^{-1}(\\hat K\\,\\odot\\,\\mathcal{F}(\\rho_c))$\nfor each of the 4 charges")
box(7.7,GY,1.8,GH,STG["spread"],5,"Gather","interpolate the node\npotentials back to\nthe points")
rbox(5.15,5.08,1.9,0.36,tint(STG["kernel"]),STG["kernel"],lw=1.4,r=0.08); badge(5.38,5.26,STG["kernel"],2,0.12)
ax.text(6.3,5.26,"kernel spectrum $\\hat K$",fontsize=8.8,fontweight="bold",color=INK,ha="center",va="center",zorder=5)
arrow((6.1,5.08),(6.1,GY+GH),col=STG["kernel"],lw=1.4)
arrow((2.1,GY+GH/2),(2.7,GY+GH/2)); arrow((4.5,GY+GH/2),(5.1,GY+GH/2)); arrow((7.1,GY+GH/2),(7.7,GY+GH/2))
lab(4.8,GY+GH/2+0.33,"4 grids\n$\\rho_c$"); lab(7.4,GY+GH/2+0.33,"4 grids\n$\\Phi_c$")
# point band
PY,PH=1.2,1.35
rbox(0.35,PY,1.75,PH,"#e6e5e1",INK,lw=1.8)
ax.text(1.225,PY+PH-0.22,"Embedding $y$",fontsize=10.5,fontweight="bold",color=INK,ha="center",va="center",zorder=5)
rng=np.random.default_rng(5); cl=[(-0.45,0.05),(0.25,0.2),(0.1,-0.1)]
for k,(cx,cy) in enumerate(cl):
    p=rng.normal(size=(14,2))*np.array([0.17,0.11])+np.array([cx,cy])
    for a,b in p: ax.add_patch(Circle((1.225+a*1.0,PY+0.6+b),0.03,fc=[STG["fft"],STG["attr"],STG["spread"]][k],ec="none",alpha=0.9,zorder=5))
ax.text(1.225,PY+0.2,"N points in 2D",fontsize=8.8,color=INK,ha="center",va="center",zorder=5)
box(2.7,PY,1.8,PH,STG["attr"],7,"Attraction","$F^{attr}_i=\\sum_j p_{ij}w_{ij}(y_i-y_j)$\nover the non-zeros of $P$")
box(7.7,PY,1.8,PH,STG["update"],6,"Assemble","$Z$ and $F^{rep}$ from the\n4 potentials at each point")
box(9.8,PY,1.25,PH,STG["update"],8,"Update","gradient, gains,\nmomentum step\n(+ re-center, 9)",sfs=8.2)
ax.text(10.475,PY+0.34,"",fontsize=8)
rbox(2.8,0.2,1.6,0.42,"white",STG["attr"],lw=1.3,ls=(0,(4,2)),r=0.08); lab(3.6,0.41,"$P$: sparse, $NK$ non-zeros",col=INK,fs=8.4)
arrow((3.6,0.62),(3.6,PY),col=STG["attr"],lw=1.4)
# links between the bands
arrow((1.225,PY+PH),(1.225,GY)); lab(1.38,2.8,"bounding box",ha="left")
arrow((2.1,PY+PH),(2.95,GY),rad=-0.12); lab(2.45,2.8,"points $y_j$ and charges",ha="left")
arrow((8.6,GY),(8.6,PY+PH)); lab(8.75,2.8,"$\\phi_c(y_i)$",ha="left")
arrow((2.1,PY+PH/2),(2.7,PY+PH/2),col=STG["attr"])
arrow((9.5,PY+PH/2),(9.8,PY+PH/2)); lab(9.65,PY+PH/2+0.22,"$F^{rep}$")
# F_attr corridor to Update (under the boxes)
CY=0.85
ax.plot([4.5,5.0,5.0,10.2],[PY+0.35,PY+0.35,CY,CY],color=STG["attr"],lw=1.6,zorder=2,solid_joinstyle="round")
arrow((10.2,CY),(10.2,PY),col=STG["attr"]); lab(7.6,CY+0.17,"$F^{attr}$ (independent of the grid)",col=STG["attr"])
# next iteration loop along the bottom
LY=0.12
ax.plot([10.75,10.75,1.225],[PY,LY,LY],color=INK,lw=1.3,zorder=1,solid_joinstyle="round")
arrow((1.225,LY),(1.225,PY),col=INK,lw=1.3)
ax.text(7.6,LY+0.17,"updated $y$ starts the next iteration ($T=1000$ in total)",fontsize=8.4,color=INK,ha="center",va="center",zorder=6)
import os; _o = f"{FIGS}/R_dataflow.png"; os.makedirs(os.path.dirname(_o) or ".", exist_ok=True)
fig.tight_layout(pad=0.3); fig.savefig(_o,dpi=200); print("ok")